# Лексический baseline

Цель - построить первый базовый метод кандидатогенерации и получить начальное значение Recall@50, относительно которого будут оцениваться последующие улучшения. В качестве baseline используется лексический поиск на основе **TF-IDF**:

- запрос представляется текстом search_query
- объявление представляется его заголовком item_title_raw
- запросы и объявления преобразуются в TF-IDF-векторы
- близость между ними вычисляется с помощью cosine similarity
- для каждого поискового контекста выбираются 50 наиболее похожих объявлений

На данном этапе намеренно не используются дополнительные признаки, такие как описание объявления, параметры, локация или история взаимодействий. Это позволяет оценить качество простого текстового retrieval и использовать его как отправную точку для следующих экспериментов.

In [18]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from scipy.sparse import vstack
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.auto import tqdm

## Загрузка данных
Для корректного сравнения с последующими экспериментами используется та же схема локальной валидации, которая была определена ранее. Сначала загрузим исходные данные и импортируем необходимые библиотеки.

In [19]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.metrics import mean_recall_at_k

In [20]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
benchmark_items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")

print("Train:", train.shape)
print("Benchmark items:", benchmark_items.shape)

Train: (497673, 19)
Benchmark items: (189212, 14)


## Воспроизведение validation split

Используется тот же `RANDOM_STATE` и тот же способ разделения, что и в validation.ipynb. Validation строится на уровне уникальных текстов `search_query`: выбранные запросы полностью исключаются из `train_retrieval`, что соответствует строгому unseen-query сценарию

In [21]:
QUERY_COLS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]

train = train.copy()

train["context_id"] = (
    train
    .groupby(QUERY_COLS, sort=False)
    .ngroup()
)

RANDOM_STATE = 42
VALID_SIZE = 2452

unique_queries = train["search_query"].drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)

valid_query_texts = set(
    rng.choice(
        unique_queries,
        size=VALID_SIZE,
        replace=False,
    )
)

is_valid = train["search_query"].isin(valid_query_texts)

train_retrieval = train.loc[~is_valid].copy()
valid_interactions = train.loc[is_valid].copy()

In [22]:
validation_queries = (
    valid_interactions[
        ["context_id"] + QUERY_COLS
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

validation_labels = (
    valid_interactions
    .groupby("context_id")["item_id"]
    .agg(lambda x: list(dict.fromkeys(x)))
    .to_dict()
)

print("Validation contexts:", len(validation_queries))

Validation contexts: 11476


## Формирование корпуса объявлений

Используется тот же validation corpus, что и при построении локальной валидации: настоящий benchmark-корпус дополняется положительными объявлениями validation. Таким образом поиск выполняется среди примерно 200 тысяч объявлений, что сопоставимо с размером реального benchmark

In [23]:
ITEM_COLS = [
    "item_title_raw",
    "item_rating_reviews_count",
    "item_rating",
    "item_price",
    "item_microcat_id",
    "item_longitude",
    "item_location_id",
    "item_latitude",
    "item_is_phone_hidden",
    "item_is_message_forbidden",
    "item_infm_params_text",
    "item_id",
    "item_description_raw",
    "item_category_id",
]

validation_positive_items = (
    valid_interactions[ITEM_COLS]
    .drop_duplicates(subset="item_id")
)

validation_items = pd.concat(
    [
        benchmark_items,
        validation_positive_items,
    ],
    ignore_index=True,
)

validation_items = (
    validation_items
    .drop_duplicates(subset="item_id")
    .reset_index(drop=True)
)

print("Validation corpus:", len(validation_items))

Validation corpus: 201651


## Функция лексического retrieval
Для проведения нескольких экспериментов вынесем TF-IDF retrieval в отдельную функцию. Функция получает текстовые представления запросов и объявлений, строит TF-IDF-векторы и возвращает до 50 наиболее похожих объявлений для каждого поискового контекста. Вычисление similarity выполняется батчами, чтобы не создавать полную плотную матрицу сходства между всеми запросами и объявлениями.

In [50]:
def tfidf_retrieval(
    query_texts,
    item_texts,
    context_ids,
    item_ids,
    top_k=50,
    batch_size=128,
    ngram_range=(1, 2),
    min_df=2,
    analyzer="word",
):
    # Выполняет TF-IDF retrieval и возвращает Top-K item_id

    vectorizer = TfidfVectorizer(
        lowercase=True,
        analyzer=analyzer,
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=0.95,
        sublinear_tf=True,
        dtype=np.float32,
    )

    item_matrix = vectorizer.fit_transform(item_texts)
    query_matrix = vectorizer.transform(query_texts)

    predictions = {}

    for start in tqdm(
        range(0, query_matrix.shape[0], batch_size),
        desc=f"TF-IDF ({analyzer}) retrieval",
    ):
        end = min(start + batch_size, query_matrix.shape[0])

        similarities = query_matrix[start:end] @ item_matrix.T

        for local_idx in range(similarities.shape[0]):
            global_idx = start + local_idx
            context_id = context_ids[global_idx]

            row = similarities.getrow(local_idx)

            if row.nnz == 0:
                predictions[context_id] = []
                continue

            scores = row.data
            indices = row.indices

            if len(scores) > top_k:
                top_positions = np.argpartition(
                    scores, -top_k
                )[-top_k:]
            else:
                top_positions = np.arange(len(scores))

            top_positions = top_positions[
                np.argsort(scores[top_positions])[::-1]
            ]

            predictions[context_id] = (
                item_ids[indices[top_positions]].tolist()
            )

    coverage = np.asarray(
        query_matrix.getnnz(axis=1) > 0
    ).ravel().mean()

    return predictions, vectorizer, coverage

## Подготовка текста

Первый эксперимент использует только:
- `search_query` для представления запроса
- `item_title_raw` для представления объявления

На данном этапе применяется минимальная предобработка: пропущенные значения заменяются пустыми строками, текст переводится в нижний регистр. Дополнительная нормализация намеренно не используется, чтобы сначала измерить качество наиболее простого решения.

In [25]:
query_texts = (
    validation_queries["search_query"]
    .fillna("")
    .astype(str)
    .str.lower()
)

item_texts = (
    validation_items["item_title_raw"]
    .fillna("")
    .astype(str)
    .str.lower()
)

print("Queries:", len(query_texts))
print("Items:", len(item_texts))

display(query_texts.head())
display(item_texts.head())

Queries: 11476
Items: 201651


0                   строительная бригада
1                     трансфер краснодар
2    гидрообразивная резка керамогранита
3                          перманент губ
4                            hydrafacial
Name: search_query, dtype: object

0    ремонт/выкуп компьют. и ноутбуков с выездом на дом
1                               обучение ребенка чтению
2                                          афрокудри 5+
3                       монтаж малых архитектурных форм
4                     репетитор по истории 10 класс егэ
Name: item_title_raw, dtype: object

## Построение TF-IDF представления
TF-IDF позволяет представить тексты в виде разреженных векторов, увеличивая вес слов, характерных для конкретного документа, и уменьшая влияние часто встречающихся слов. Для первого baseline используются униграммы и биграммы слов.

Векторизатор обучается на корпусе объявлений, после чего тем же преобразованием кодируются поисковые запросы.

In [26]:
tfidf = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32,
)

In [27]:
item_matrix = tfidf.fit_transform(item_texts)
query_matrix = tfidf.transform(query_texts)

print("Размер словаря:", len(tfidf.vocabulary_))
print("Item matrix:", item_matrix.shape)
print("Query matrix:", query_matrix.shape)

Размер словаря: 74575
Item matrix: (201651, 74575)
Query matrix: (11476, 74575)


## Покрытие словаря
Лексический retrieval способен использовать только те слова и словосочетания, которые присутствуют в словаре TF-IDF. Проверим долю validation-запросов, для которых после преобразования получился непустой вектор.

In [28]:
query_has_features = np.asarray(
    query_matrix.getnnz(axis=1) > 0
).ravel()

coverage = query_has_features.mean()

print(f"Запросов с непустым TF-IDF-вектором: {coverage:.2%}")
print(f"Запросов без известных токенов: {(~query_has_features).sum()}")

Запросов с непустым TF-IDF-вектором: 99.56%
Запросов без известных токенов: 50


## Поиск Top-50 кандидатов
TF-IDF по умолчанию использует L2-нормализацию векторов. Поэтому скалярное произведение TF-IDF-векторов соответствует cosine similarity. Чтобы не хранить матрицу сходства всех запросов со всеми объявлениями одновременно, retrieval выполняется батчами.

Для каждого поискового контекста сохраняются только 50 объявлений с максимальным значением similarity.

In [29]:
TOP_K = 50
BATCH_SIZE = 128

item_ids = validation_items["item_id"].to_numpy()
context_ids = validation_queries["context_id"].to_numpy()

predictions = {}

In [30]:
for start in tqdm(
    range(0, query_matrix.shape[0], BATCH_SIZE),
    desc="TF-IDF retrieval",
):
    end = min(start + BATCH_SIZE, query_matrix.shape[0])

    batch_queries = query_matrix[start:end]

    similarities = batch_queries @ item_matrix.T

    for local_idx in range(similarities.shape[0]):
        global_idx = start + local_idx
        context_id = context_ids[global_idx]

        row = similarities.getrow(local_idx)

        if row.nnz == 0:
            predictions[context_id] = []
            continue

        scores = row.data
        indices = row.indices

        if len(scores) > TOP_K:
            top_positions = np.argpartition(
                scores,
                -TOP_K,
            )[-TOP_K:]
        else:
            top_positions = np.arange(len(scores))

        top_positions = top_positions[
            np.argsort(scores[top_positions])[::-1]
        ]

        top_item_indices = indices[top_positions]

        predictions[context_id] = (
            item_ids[top_item_indices].tolist()
        )

TF-IDF retrieval:   0%|          | 0/90 [00:00<?, ?it/s]

### Проверка

In [31]:
print("Получено predictions:", len(predictions))
print("Validation contexts:", len(validation_queries))

assert len(predictions) == len(validation_queries)

Получено predictions: 11476
Validation contexts: 11476


In [32]:
prediction_lengths = pd.Series(
    [len(items) for items in predictions.values()]
)

display(prediction_lengths.describe())

print(
    "Контекстов с 50 кандидатами:",
    f"{(prediction_lengths == 50).mean():.2%}"
)

print(
    "Контекстов без кандидатов:",
    (prediction_lengths == 0).sum()
)

count    11476.000000
mean        47.986494
std          8.729610
min          0.000000
25%         50.000000
50%         50.000000
75%         50.000000
max         50.000000
dtype: float64

Контекстов с 50 кандидатами: 94.17%
Контекстов без кандидатов: 50


## Оценка Recall@50
Полученные кандидаты сравниваются с ground truth локальной validation. Recall@50 показывает какая доля известных релевантных объявлений в среднем попала в набор из 50 кандидатов для каждого поискового контекста.

In [33]:
recall_50 = mean_recall_at_k(
    predictions=predictions,
    labels=validation_labels,
    k=50,
)

print(f"TF-IDF title Recall@50: {recall_50:.4f}")

TF-IDF title Recall@50: 0.1880


In [34]:
# посмотрим распределение
from src.metrics import recall_at_k

recall_rows = []

for context_id, relevant_items in validation_labels.items():
    predicted_items = predictions.get(context_id, [])

    recall = recall_at_k(
        predicted_items,
        relevant_items,
        k=50,
    )

    recall_rows.append(
        {
            "context_id": context_id,
            "relevant_count": len(relevant_items),
            "predicted_count": len(predicted_items),
            "recall": recall,
        }
    )

recall_df = pd.DataFrame(recall_rows)

display(recall_df["recall"].describe())

print(
    "Контекстов с Recall = 1:",
    f"{(recall_df['recall'] == 1).mean():.2%}"
)

print(
    "Контекстов с Recall = 0:",
    f"{(recall_df['recall'] == 0).mean():.2%}"
)

count    11476.000000
mean         0.187956
std          0.380111
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          1.000000
Name: recall, dtype: float64

Контекстов с Recall = 1: 17.21%
Контекстов с Recall = 0: 79.27%


In [35]:
# конкретные ошибки
failed_contexts = recall_df.loc[
    recall_df["recall"] == 0,
    "context_id",
].head(10)

for context_id in failed_contexts:
    query_row = validation_queries[
        validation_queries["context_id"] == context_id
    ].iloc[0]

    relevant_ids = validation_labels[context_id]

    relevant_titles = validation_items[
        validation_items["item_id"].isin(relevant_ids)
    ]["item_title_raw"].tolist()

    predicted_titles = validation_items[
        validation_items["item_id"].isin(
            predictions.get(context_id, [])[:5]
        )
    ]["item_title_raw"].tolist()

    print("=" * 80)
    print("Запрос:", query_row["search_query"])
    print("Relevant:", relevant_titles)
    print("Top candidates:", predicted_titles)

Запрос: строительная бригада
Relevant: ['Бригада строителей, все виды строительных услуг']
Top candidates: ['Строительная бригада', 'Строительная бригада', 'Строительная бригада', 'Строительная бригада', 'Строительная бригада']
Запрос: трансфер краснодар
Relevant: ['От 400 км Такси межгород Трансфер без попутчиков']
Top candidates: ['Трансфер Краснодар — Тбилиси — Ереван — Батуми', 'Трансфер Краснодар минивэн –море,аэропорт,межгород', 'Трансфер краснодар-москва-краснодар', 'Трансфер Краснодар минивэн город межгород 6 мест', 'Такси трансфер Краснодар Сочи межгород аэропорты']
Запрос: гидрообразивная резка керамогранита
Relevant: ['Профессиональная резка плитки и керамогранита']
Top candidates: ['Резка керамогранита', 'Резка керамогранита', 'Резка керамогранита', 'Резка керамогранита', 'Резка керамогранита']
Запрос: перманент губ
Relevant: ['Перманентный макияж (перманент, татуаж)']
Top candidates: ['Требуются модели на перманент губ', 'Перманент губ,бровей,глаз,стрелок,биоревитализация'

## Выводы для первого эксперемента

Первый лексический baseline на основе TF-IDF по `search_query` и `item_title_raw` показал Recall@50 = 0.1880.
Основные наблюдения:

- 99,56% validation-запросов имеют непустое TF-IDF представление, поэтому низкий Recall практически не связан с недостаточным покрытием словаря;
- для 94,17% поисковых контекстов лексический retrieval формирует полный набор из 50 кандидатов
- только 17,21% контекстов имеют Recall = 1, тогда как для 79,27% контекстов ни одно релевантное объявление не попадает в Top-50
- анализ ошибок показывает, что заголовка объявления часто недостаточно для определения релевантности: информация, соответствующая запросу, может содержаться в описании или дополнительных параметрах
- в корпусе присутствует большое количество объявлений с одинаковыми или очень похожими заголовками. В таких случаях TF-IDF практически не способен различить конкретные item_id
- следовательно, дальнейшее улучшение должно быть направлено прежде всего на повышение покрытия релевантных объявлений в candidate set, а не на точное ранжирование уже найденных кандидатов

Следующим этапом будет расширение текстового представления объявления с использованием `item_description_raw` и `item_infm_params_text`, после чего будет исследовано использование локации как дополнительного сигнала

## Эксперимент 2 - заголовок + параметры объявления

В первом эксперименте использовался только `item_title_raw`. Однако часть информации об услуге может находиться в `item_infm_params_text`. Поэтому расширим текстовое представление объявления: `item_title_raw + item_infm_params_text` Остальные параметры TF-IDF и validation остаются неизменными, чтобы изменение `Recall@50` можно было связать именно с добавлением нового текстового признака.

In [36]:
display(
    validation_items[
        ["item_title_raw", "item_infm_params_text"]
    ].head(10)
)

,item_title_raw,item_infm_params_text
0,Ремонт/выкуп компьют. и ноутбуков с выездом на дом,Вид услуги Компьютерная помощь Место оказания услуг пр-т Ленина Тип стоимости за услугу Начальная цена График работы от 34200 График работы до 828...
1,Обучение ребенка чтению,"Вид услуги Обучение, курсы Место оказания услуг ул. Чернышевского, 35 Тип услуги Детское развитие, логопеды Тип стоимости за урок Начальная цена Г..."
2,Афрокудри 5+,"Вид услуги Красота, здоровье Место оказания услуг городской округ Иваново, Фрунзенский район Тип услуги Услуги парикмахера Тип стоимости за услугу..."
3,Монтаж малых архитектурных форм,"Вид услуги Строительство Место оказания услуг Севастопольский пр-т, 28к4 Тип стоимости за услугу Работа по договору Гарантия График работы от 2880..."
4,Репетитор по истории 10 класс ЕГЭ,"Вид услуги Обучение, курсы Место оказания услуг Волгоградская обл., Волжский, пл. имени В. И. Ленина Тип услуги Предметы школы и вуза Начальная це..."
5,Ремонт посудомоечных машин с выездом на дом,"Вид услуги Ремонт и обслуживание техники Тип услуги Посудомоечные машины Место оказания услуг Москва, район Щукино График работы от 28800 График р..."
6,Подарок на праздник - Песня на заказ,"Вид услуги Искусство Место оказания услуг Чувашская Республика — Чувашия, Новочебоксарск"
7,Пух перо,"Вид услуги Вывоз мусора и вторсырья Место оказания услуг Ставропольский край, Будённовск, Пушкинская улица Тип стоимости за услугу График работы о..."
8,Металлолом,"Вид услуги Другое Место оказания услуг ул. Новосёлов, 33 Тип стоимости за услугу Куда выезжаете Не выезжаю"
9,Мотопомпа Fubag бензиновая в аренду,"Вид услуги Оборудование, производство Тип услуги Аренда оборудования Место оказания услуг Липецкая обл., Липецк Тип стоимости за услугу Начальная ..."


In [37]:
# строим текст
item_texts_b2 = (
    validation_items["item_title_raw"]
    .fillna("")
    .astype(str)
    + " "
    + validation_items["item_infm_params_text"]
    .fillna("")
    .astype(str)
).str.lower()

In [38]:
query_texts = (
    validation_queries["search_query"]
    .fillna("")
    .astype(str)
    .str.lower()
)

In [39]:
# запускаем
predictions_b2, tfidf_b2, coverage_b2 = tfidf_retrieval(
    query_texts=query_texts,
    item_texts=item_texts_b2,
    context_ids=validation_queries["context_id"].to_numpy(),
    item_ids=validation_items["item_id"].to_numpy(),
)

TF-IDF retrieval:   0%|          | 0/90 [00:00<?, ?it/s]

In [40]:
# считаем
recall_b2 = mean_recall_at_k(
    predictions=predictions_b2,
    labels=validation_labels,
    k=50,
)

print(f"Coverage: {coverage_b2:.2%}")
print(f"B1 Recall@50: {recall_50:.4f}")
print(f"B2 Recall@50: {recall_b2:.4f}")
print(f"Изменение: {recall_b2 - recall_50:+.4f}")

Coverage: 99.73%
B1 Recall@50: 0.1880
B2 Recall@50: 0.1882
Изменение: +0.0003


## Посмотрим Recall=0 и Recall=1 для 2

In [41]:
def evaluate_predictions(predictions, labels):
    # Возвращает Recall для каждого поискового контекста

    rows = []

    for context_id, relevant_items in labels.items():
        predicted_items = predictions.get(context_id, [])

        recall = recall_at_k(
            predicted_items,
            relevant_items,
            k=50,
        )

        rows.append({
            "context_id": context_id,
            "relevant_count": len(relevant_items),
            "predicted_count": len(predicted_items),
            "recall": recall,
        })

    return pd.DataFrame(rows)

In [42]:
recall_b2_df = evaluate_predictions(
    predictions_b2,
    validation_labels,
)

print(
    "Recall = 1:",
    f"{(recall_b2_df['recall'] == 1).mean():.2%}"
)

print(
    "Recall = 0:",
    f"{(recall_b2_df['recall'] == 0).mean():.2%}"
)

Recall = 1: 17.31%
Recall = 0: 79.36%


## Эксперимент 3 - заголовок + параметры + описание
Следующий эксперимент дополнительно использует `item_description_raw`. Описание содержит более подробную информацию об услуге и потенциально позволяет находить релевантные объявления даже в случаях, когда формулировка запроса отсутствует в заголовке.

Текстовое представление объявления:`item_title_raw + item_infm_params_text + item_description_raw`

In [43]:
# создаем
item_texts_b3 = (
    validation_items["item_title_raw"]
    .fillna("")
    .astype(str)
    + " "
    + validation_items["item_infm_params_text"]
    .fillna("")
    .astype(str)
    + " "
    + validation_items["item_description_raw"]
    .fillna("")
    .astype(str)
).str.lower()

In [44]:
# запускаем
predictions_b3, tfidf_b3, coverage_b3 = tfidf_retrieval(
    query_texts=query_texts,
    item_texts=item_texts_b3,
    context_ids=validation_queries["context_id"].to_numpy(),
    item_ids=validation_items["item_id"].to_numpy(),
)

TF-IDF retrieval:   0%|          | 0/90 [00:00<?, ?it/s]

In [45]:
# оцениваем
recall_b3 = mean_recall_at_k(
    predictions=predictions_b3,
    labels=validation_labels,
    k=50,
)

print(f"B1 title:                    {recall_50:.4f}")
print(f"B2 title + params:           {recall_b2:.4f}")
print(f"B3 title + params + desc:    {recall_b3:.4f}")

B1 title:                    0.1880
B2 title + params:           0.1882
B3 title + params + desc:    0.2328


## Таблица экспериментов

In [46]:
results = pd.DataFrame([
    {
        "experiment": "B1",
        "item_text": "title",
        "extra_features": "—",
        "recall@50": recall_50,
    },
    {
        "experiment": "B2",
        "item_text": "title + params",
        "extra_features": "—",
        "recall@50": recall_b2,
    },
    {
        "experiment": "B3",
        "item_text": "title + params + description",
        "extra_features": "—",
        "recall@50": recall_b3,
    },
])

display(results)

,experiment,item_text,extra_features,recall@50
0,B1,title,—,0.187956
1,B2,title + params,—,0.188225
2,B3,title + params + description,—,0.232802


## Эксперимент 4 - заголовок + описание
Предыдущий эксперимент показал, что добавление `item_infm_params_text` практически не изменяет качество retrieval, тогда как добавление описания приводит к заметному росту Recall@50. Чтобы отдельно оценить вклад описания и проверить, не добавляют ли параметры лишний текстовый шум, протестируем представление: `item_title_raw + item_description_raw`

In [47]:
item_texts_b4 = (
    validation_items["item_title_raw"]
    .fillna("")
    .astype(str)
    + " "
    + validation_items["item_description_raw"]
    .fillna("")
    .astype(str)
).str.lower()

In [48]:
# запускаем
predictions_b4, tfidf_b4, coverage_b4 = tfidf_retrieval(
    query_texts=query_texts,
    item_texts=item_texts_b4,
    context_ids=validation_queries["context_id"].to_numpy(),
    item_ids=validation_items["item_id"].to_numpy(),
)

TF-IDF retrieval:   0%|          | 0/90 [00:00<?, ?it/s]

In [49]:
recall_b4 = mean_recall_at_k(
    predictions=predictions_b4,
    labels=validation_labels,
    k=50,
)

print(f"B1 title:                       {recall_50:.4f}")
print(f"B2 title + params:              {recall_b2:.4f}")
print(f"B3 title + params + desc:       {recall_b3:.4f}")
print(f"B4 title + description:         {recall_b4:.4f}")

B1 title:                       0.1880
B2 title + params:              0.1882
B3 title + params + desc:       0.2328
B4 title + description:         0.2131


## Сравнение текстовых представлений
Эксперименты показывают, что использование только заголовка объявления недостаточно для качественного candidate generation.

- `title` даёт Recall@50 = 0.1880
- добавление только параметров практически не изменяет результат (`0.1882`)
- добавление описания повышает Recall до `0.2131`
- наилучший результат достигается при совместном использовании заголовка, параметров и описания Recall@50 = 0.2328

Таким образом, `item_description_raw` содержит существенную дополнительную информацию о предоставляемой услуге. Поле `item_infm_params_text` само по себе почти не улучшает retrieval, однако в комбинации с описанием даёт дополнительный прирост качества.

В качестве основного текстового представления объявления для следующих экспериментов выбирается: `item_title_raw + item_infm_params_text + item_description_raw`

## Эксперимент 5 - Character TF-IDF
Предыдущие эксперименты использовали TF-IDF на уровне слов. Такой подход чувствителен к опечаткам, различным словоформам и вариантам написания. Для русского языка эти различия могут быть особенно значимы из-за развитой морфологии.

В данном эксперименте используется Character TF-IDF с символьными n-граммами длины 3–5 символов. В качестве текста объявления используется лучшее представление из предыдущих экспериментов: `title + params + description`

Character TF-IDF будет оценён как отдельный retrieval-канал, после чего его кандидаты можно будет объединить с word-level TF-IDF.

In [51]:
predictions_b5, tfidf_b5, coverage_b5 = tfidf_retrieval(
    query_texts=query_texts,
    item_texts=item_texts_b3,
    context_ids=validation_queries["context_id"].to_numpy(),
    item_ids=validation_items["item_id"].to_numpy(),
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
)

TF-IDF (char_wb) retrieval:   0%|          | 0/90 [00:00<?, ?it/s]

In [52]:
# считаем
recall_b5 = mean_recall_at_k(
    predictions=predictions_b5,
    labels=validation_labels,
    k=50,
)

print(f"B3 Word TF-IDF:       {recall_b3:.4f}")
print(f"B5 Character TF-IDF:  {recall_b5:.4f}")
print(f"Изменение:             {recall_b5 - recall_b3:+.4f}")
print(f"Coverage B5:           {coverage_b5:.2%}")

B3 Word TF-IDF:       0.2328
B5 Character TF-IDF:  0.2240
Изменение:             -0.0088
Coverage B5:           100.00%


## Эксперимент 6 - TF-IDF + location boost
По результатам EDA примерно 83% положительных взаимодействий имеют совпадающие `search_location_id` и `item_location_id`. Локация является сильным дополнительным сигналом релевантности, особенно для запросов с большим количеством лексически похожих объявлений.

При этом локацию нельзя использовать как жёсткий фильтр, поскольку часть релевантных объявлений находится в другой локации. Поэтому используется мягкий location boost: `final_score = tfidf_score + α · location_match`, где `location_match = 1`, если локации запроса и объявления совпадают, и `0` иначе. Проверим несколько значений коэффициента α и выберем лучшее по validation Recall@50.

In [53]:
def tfidf_retrieval_with_location(
    query_texts,
    item_texts,
    query_locations,
    item_locations,
    context_ids,
    item_ids,
    location_boost=0.1,
    top_k=50,
    candidate_k=500,
    batch_size=128,
):
    # TF-IDF retrieval с дополнительным boost за совпадение локации.

    # Сначала для каждого запроса получаем расширенный набор
    # лексических кандидатов, после чего добавляем bonus объявлениям
    # из той же локации и выбираем итоговый Top-K

    vectorizer = TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        dtype=np.float32,
    )

    # Обучаем TF-IDF на текстах объявлений
    item_matrix = vectorizer.fit_transform(item_texts)

    # Преобразуем поисковые запросы
    query_matrix = vectorizer.transform(query_texts)

    predictions = {}

    for start in tqdm(
        range(0, query_matrix.shape[0], batch_size),
        desc=f"TF-IDF + location ({location_boost})",
    ):
        end = min(start + batch_size, query_matrix.shape[0])

        # Лексическое сходство запросов и объявлений
        similarities = query_matrix[start:end] @ item_matrix.T

        for local_idx in range(similarities.shape[0]):
            global_idx = start + local_idx
            context_id = context_ids[global_idx]

            row = similarities.getrow(local_idx)

            if row.nnz == 0:
                predictions[context_id] = []
                continue

            scores = row.data
            indices = row.indices

            # Сначала оставляем более широкий набор lexical candidates.
            # Это позволяет location boost поднять объявления,
            # которые находились ниже исходного Top-50.
            if len(scores) > candidate_k:
                candidate_positions = np.argpartition(
                    scores,
                    -candidate_k,
                )[-candidate_k:]
            else:
                candidate_positions = np.arange(len(scores))

            candidate_indices = indices[candidate_positions]

            candidate_scores = scores[candidate_positions].copy()

            # Локация текущего запроса
            query_location = query_locations[global_idx]

            # Локации найденных объявлений
            candidate_locations = item_locations[candidate_indices]

            # Совпадение локации
            location_match = (
                candidate_locations == query_location
            ).astype(np.float32)

            # Добавляем мягкий location boost
            final_scores = (
                candidate_scores
                + location_boost * location_match
            )

            # Выбираем итоговый Top-50
            if len(final_scores) > top_k:
                top_positions = np.argpartition(
                    final_scores,
                    -top_k,
                )[-top_k:]
            else:
                top_positions = np.arange(len(final_scores))

            # Сортируем кандидатов по итоговому score
            top_positions = top_positions[
                np.argsort(final_scores[top_positions])[::-1]
            ]

            final_indices = candidate_indices[top_positions]

            predictions[context_id] = (
                item_ids[final_indices].tolist()
            )

    return predictions

In [54]:
# Подготавливаем массивы
query_locations = (
    validation_queries["search_location_id"].to_numpy()
)

item_locations = (
    validation_items["item_location_id"].to_numpy()
)

context_ids = (
    validation_queries["context_id"].to_numpy()
)

item_ids = (
    validation_items["item_id"].to_numpy()
)

In [55]:
# проверяем location boost
location_results = []

for alpha in [0.05, 0.10, 0.20]:
    
    predictions_location = tfidf_retrieval_with_location(
        query_texts=query_texts,
        item_texts=item_texts_b3,
        query_locations=query_locations,
        item_locations=item_locations,
        context_ids=context_ids,
        item_ids=item_ids,
        location_boost=alpha,
        top_k=50,
        candidate_k=500,
        batch_size=128,
    )

    recall_location = mean_recall_at_k(
        predictions=predictions_location,
        labels=validation_labels,
        k=50,
    )

    location_results.append({
        "location_boost": alpha,
        "recall@50": recall_location,
    })

    print(
        f"alpha={alpha:.2f} | "
        f"Recall@50={recall_location:.4f}"
    )

TF-IDF + location (0.05):   0%|          | 0/90 [00:00<?, ?it/s]

alpha=0.05 | Recall@50=0.4183


TF-IDF + location (0.1):   0%|          | 0/90 [00:00<?, ?it/s]

alpha=0.10 | Recall@50=0.5087


TF-IDF + location (0.2):   0%|          | 0/90 [00:00<?, ?it/s]

alpha=0.20 | Recall@50=0.5392


In [56]:
location_results_df = pd.DataFrame(location_results)

display(location_results_df)

,location_boost,recall@50
0,0.05,0.418308
1,0.10,0.508675
2,0.20,0.539246


In [57]:
best_location_result = (
    location_results_df
    .sort_values("recall@50", ascending=False)
    .iloc[0]
)

best_alpha = best_location_result["location_boost"]
best_location_recall = best_location_result["recall@50"]

print(f"B3 TF-IDF:              {recall_b3:.4f}")
print(f"Best location boost:    {best_alpha:.2f}")
print(f"B6 TF-IDF + location:   {best_location_recall:.4f}")
print(
    f"Изменение:              "
    f"{best_location_recall - recall_b3:+.4f}"
)

B3 TF-IDF:              0.2328
Best location boost:    0.20
B6 TF-IDF + location:   0.5392
Изменение:              +0.3064


### Итоги по lexical retrieval
Наибольший прирост качества дал учёт географического контекста запроса. Базовый Word TF-IDF по расширенному тексту объявления (`title + params + description`) показывает Recall@50 = 0.2328.

Добавление мягкого boost за совпадение `search_location_id` и `item_location_id` значительно повышает качество:

- α = 0.05 -> `Recall@50 = 0.4183`
- α = 0.10 -> `Recall@50 = 0.5087`
- α = 0.20 -> `Recall@50 = 0.5392`

Лучший результат получен при `α = 0.20`.

Это подтверждает вывод EDA о высокой значимости географического признака: большинство положительных взаимодействий происходит между запросом и объявлением из одной локации. При этом location используется как дополнительный сигнал, а не как жёсткий фильтр, что позволяет сохранять объявления из других локаций среди кандидатов.

В качестве финального lexical candidate generator выбирается Word TF-IDF по `title + params + description` с location boost `α = 0.20`.